## 4.4 search evaluation

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

df_ground_truth=pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
df_ground_truth

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf
...,...,...
390,Why do I get a 401 Client Error when using the...,4b30b918bc
391,What's the easiest way to force-install reques...,4b30b918bc
392,Can I install requests straight from the GitHu...,4b30b918bc
393,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc


In [4]:
df_ground_truth_data = pd.read_csv('data/ground-truth-data.csv')
df_ground_truth_data.head()

,question,course,document
0,Can I take this course at my own pace and stil...,llm-zoomcamp,69d122f12e
1,Is a certificate available if I complete the c...,llm-zoomcamp,69d122f12e
2,Do self-paced learners get any certificate for...,llm-zoomcamp,69d122f12e
3,Why are certificates not issued for the self-p...,llm-zoomcamp,69d122f12e
4,Is peer review of capstone projects required i...,llm-zoomcamp,69d122f12e


In [5]:
df_ground_truth_data.shape

(395, 3)

In [6]:
from part_1_RAG.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [8]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [9]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [10]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')


74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False
489dd1c9d9 == 74eb249bbf: False


In [11]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [12]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [13]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [14]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [15]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [16]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [17]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [18]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [19]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [20]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)

  0%|          | 0/15 [00:00<?, ?it/s]

In [21]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

## 4.5 Search Evaluation Metrics

### Hit Rate

In [22]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt+=1
    return cnt / len(relevance)

In [23]:
hit_rate(relevance_total)

0.8987341772151899

### Mean Reciprocal Rank (MRR)

In [24]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1/(rank+1)
                break
    return total_score / len(relevance)

In [25]:
mrr(relevance_total)

0.7672151898734174

In [26]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [27]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.8987341772151899, 'mrr': 0.7672151898734174}

## 4.6 Search Parameter Tuning

In [28]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )   
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9113924050632911, 'mrr': 0.7978902953586497}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9240506329113924, 'mrr': 0.8126160337552741}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8987341772151899, 'mrr': 0.7672151898734174}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8708860759493671, 'mrr': 0.7383122362869197}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8582278481012658, 'mrr': 0.7118987341772152}


In [30]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [31]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [33]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.974684,0.883755
19,2.0,4.0,0.2,0.974684,0.883755
35,5.0,10.0,0.5,0.974684,0.883755
20,2.0,4.0,0.5,0.977215,0.882869
4,1.0,2.0,0.2,0.977215,0.882785
34,5.0,10.0,0.2,0.972152,0.882616
18,2.0,4.0,0.1,0.972152,0.882616
33,5.0,10.0,0.1,0.972152,0.882616
5,1.0,2.0,0.5,0.964557,0.861814
6,1.0,4.0,0.1,0.969620,0.861561


In [ ]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )